In [1]:
# ==============================================================================
# 1. PERSIAPAN DAN MUAT DATA
# ==============================================================================
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# Nama file CSV Anda
file_path = 'komen positif negatif.csv'

# Muat data dari file CSV
try:
    df = pd.read_csv(file_path)
    print("✅ Data berhasil dimuat.")
    print("Jumlah data:", len(df))
    print("\n5 baris data pertama:")
    print(df.head())
except FileNotFoundError:
    print(f"❌ ERROR: File '{file_path}' tidak ditemukan. Pastikan file sudah diupload ke sesi Colab Anda.")
    exit()

# Kolom yang akan digunakan
TEXT_COLUMN = 'text'
LABEL_COLUMN = 'Sentimen'

# Cek apakah kolom yang dibutuhkan ada
if TEXT_COLUMN not in df.columns or LABEL_COLUMN not in df.columns:
    print(f"❌ ERROR: Kolom '{TEXT_COLUMN}' atau '{LABEL_COLUMN}' tidak ditemukan dalam file CSV.")
    exit()

✅ Data berhasil dimuat.
Jumlah data: 323

5 baris data pertama:
  Sentimen       author                                               text  \
0  Positif   User_Bijak  Penyampaian yang sangat berbobot dan cerdas. S...   
1  Positif    AkbarGlow  Setuju banget sama poin-poinnya. Akhirnya ada ...   
2  Positif  Rani_Cerdas  Diskusi yang sangat mencerahkan. Saya mendapat...   
3  Positif  Tulus_Pikir  Sangat menginspirasi. Indonesia butuh lebih ba...   
4  Positif  BintangJaya  Salut buat Om Deddy dan Ferry. Pertanyaan dan ...   

    publishedAt likeCount  
0  5 months ago      1.5K  
1  5 months ago       890  
2  5 months ago      1.2K  
3  4 months ago       550  
4  4 months ago      2.1K  


In [2]:
# ==============================================================================
# 2. PREPROCESSING TEKS
# ==============================================================================

# Definisikan fungsi untuk membersihkan teks (case folding, menghilangkan tanda baca, dll.)
def clean_text(text):
    # 1. Case Folding (mengubah ke huruf kecil)
    text = text.lower()

    # 2. Menghilangkan Tanda Baca (punctuation)
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 3. Menghilangkan Angka (numerik)
    text = re.sub(r'\d+', '', text)

    # 4. Menghilangkan spasi berlebihan dan memotong spasi di awal/akhir
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)

    return text

print("\n--- Memulai Preprocessing Teks ---")
# Terapkan fungsi cleaning ke kolom 'text'
df['text_clean'] = df[TEXT_COLUMN].apply(clean_text)

print("✅ Preprocessing selesai.")
print("Contoh teks sebelum vs sesudah:")
print(f"Sebelum: {df[TEXT_COLUMN].iloc[0]}")
print(f"Sesudah: {df['text_clean'].iloc[0]}")


--- Memulai Preprocessing Teks ---
✅ Preprocessing selesai.
Contoh teks sebelum vs sesudah:
Sebelum: Penyampaian yang sangat berbobot dan cerdas. Salut dengan argumen Bang Ferry.
Sesudah: penyampaian yang sangat berbobot dan cerdas salut dengan argumen bang ferry


In [3]:
# ==============================================================================
# 3. PEMBAGIAN DATA & EKSTRAKSI FITUR (TF-IDF)
# ==============================================================================

# Pisahkan fitur (X) dan label (y)
X = df['text_clean']
y = df[LABEL_COLUMN]

# Pisahkan data menjadi data training dan data testing (misalnya 80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\n--- Pembagian Data Training/Testing (20%) ---")
print(f"Data Training: {len(X_train)} sampel")
print(f"Data Testing: {len(X_test)} sampel")

# Inisialisasi TF-IDF Vectorizer
# max_features=500 untuk membatasi jumlah fitur (kata) yang paling sering muncul
tfidf_vectorizer = TfidfVectorizer(max_features=500)

# Lakukan proses fit hanya pada data training, kemudian transform pada training dan testing
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("\n--- Ekstraksi Fitur (TF-IDF) ---")
print(f"✅ Dimensi Matriks Fitur Training: {X_train_tfidf.shape}")
print(f"✅ Jumlah Fitur (Kata) yang diekstrak: {len(tfidf_vectorizer.get_feature_names_out())}")


--- Pembagian Data Training/Testing (20%) ---
Data Training: 258 sampel
Data Testing: 65 sampel

--- Ekstraksi Fitur (TF-IDF) ---
✅ Dimensi Matriks Fitur Training: (258, 500)
✅ Jumlah Fitur (Kata) yang diekstrak: 500


In [4]:
# ==============================================================================
# 4. KLASIFIKASI DENGAN NAIVE BAYES
# ==============================================================================

print("\n--- Pelatihan Model Naive Bayes ---")
# Inisialisasi model Naive Bayes (MultinomialNB cocok untuk fitur hitungan/frekuensi seperti TF-IDF)
nb_model = MultinomialNB()

# Latih model menggunakan data training yang sudah diekstrak fiturnya
nb_model.fit(X_train_tfidf, y_train)

print("✅ Pelatihan model selesai.")

# Prediksi pada data testing
y_pred = nb_model.predict(X_test_tfidf)


--- Pelatihan Model Naive Bayes ---
✅ Pelatihan model selesai.


In [5]:
# ==============================================================================
# 5. EVALUASI MODEL
# ==============================================================================

print("\n=======================================================")
print("                   HASIL EVALUASI MODEL                ")
print("=======================================================")

# Hitung Akurasi
accuracy = accuracy_score(y_test, y_pred)
print(f"Akurasi Model: {accuracy*100:.2f}%")
print("\nLaporan Klasifikasi (Precision, Recall, F1-Score):")
print(classification_report(y_test, y_pred))


                   HASIL EVALUASI MODEL                
Akurasi Model: 87.69%

Laporan Klasifikasi (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

     Negatif       0.89      0.89      0.89        37
     Positif       0.86      0.86      0.86        28

    accuracy                           0.88        65
   macro avg       0.87      0.87      0.87        65
weighted avg       0.88      0.88      0.88        65



PERCOBAAN 1

In [7]:
# ==============================================================================
# 6. CONTOH PENGUJIAN KOMENTAR BARU
# ==============================================================================

def predict_sentiment(text_input, model, vectorizer):
    # 1. Preprocessing Teks Input
    clean_input = clean_text(text_input)
    # 2. Ekstraksi Fitur (menggunakan vectorizer yang sudah di-fit)
    vectorized_input = vectorizer.transform([clean_input])
    # 3. Prediksi
    prediction = model.predict(vectorized_input)[0]
    return prediction

print("\n--- Contoh Prediksi Komentar Baru ---")

# Komentar Positif
new_comment_positive = "Saya sangat suka dengan ketenangan dan wibawanya.."
predicted_label_pos = predict_sentiment(new_comment_positive, nb_model, tfidf_vectorizer)
print(f"Komentar: '{new_comment_positive}' -> Prediksi: {predicted_label_pos}")

# Komentar Negatif
new_comment_negative = "Cuma cari panggung biar viral, isinya tidak ada solusi konkret."
predicted_label_neg = predict_sentiment(new_comment_negative, nb_model, tfidf_vectorizer)
print(f"Komentar: '{new_comment_negative}' -> Prediksi: {predicted_label_neg}")


--- Contoh Prediksi Komentar Baru ---
Komentar: 'Saya sangat suka dengan ketenangan dan wibawanya..' -> Prediksi: Positif
Komentar: 'Cuma cari panggung biar viral, isinya tidak ada solusi konkret.' -> Prediksi: Negatif


2

In [8]:
# ==============================================================================
# 6. CONTOH PENGUJIAN KOMENTAR BARU
# ==============================================================================

def predict_sentiment(text_input, model, vectorizer):
    # 1. Preprocessing Teks Input
    clean_input = clean_text(text_input)
    # 2. Ekstraksi Fitur (menggunakan vectorizer yang sudah di-fit)
    vectorized_input = vectorizer.transform([clean_input])
    # 3. Prediksi
    prediction = model.predict(vectorized_input)[0]
    return prediction

print("\n--- Contoh Prediksi Komentar Baru ---")

# Komentar Positif
new_comment_positive = "Semangatnya untuk perubahan sangat menular. Lanjutkan!"
predicted_label_pos = predict_sentiment(new_comment_positive, nb_model, tfidf_vectorizer)
print(f"Komentar: '{new_comment_positive}' -> Prediksi: {predicted_label_pos}")

# Komentar Negatif
new_comment_negative = "Nggak enak didengar dan dilihat."
predicted_label_neg = predict_sentiment(new_comment_negative, nb_model, tfidf_vectorizer)
print(f"Komentar: '{new_comment_negative}' -> Prediksi: {predicted_label_neg}")


--- Contoh Prediksi Komentar Baru ---
Komentar: 'Semangatnya untuk perubahan sangat menular. Lanjutkan!' -> Prediksi: Positif
Komentar: 'Nggak enak didengar dan dilihat.' -> Prediksi: Negatif


3

In [9]:
# ==============================================================================
# 6. CONTOH PENGUJIAN KOMENTAR BARU
# ==============================================================================

def predict_sentiment(text_input, model, vectorizer):
    # 1. Preprocessing Teks Input
    clean_input = clean_text(text_input)
    # 2. Ekstraksi Fitur (menggunakan vectorizer yang sudah di-fit)
    vectorized_input = vectorizer.transform([clean_input])
    # 3. Prediksi
    prediction = model.predict(vectorized_input)[0]
    return prediction

print("\n--- Contoh Prediksi Komentar Baru ---")

# Komentar Positif
new_comment_positive = "Kritik yang membangun dan data yang kuat. Ini yang dinamakan aktivis sejati."
predicted_label_pos = predict_sentiment(new_comment_positive, nb_model, tfidf_vectorizer)
print(f"Komentar: '{new_comment_positive}' -> Prediksi: {predicted_label_pos}")

# Komentar Negatif
new_comment_negative = "Tidak bermanfaat sama sekali."
predicted_label_neg = predict_sentiment(new_comment_negative, nb_model, tfidf_vectorizer)
print(f"Komentar: '{new_comment_negative}' -> Prediksi: {predicted_label_neg}")


--- Contoh Prediksi Komentar Baru ---
Komentar: 'Kritik yang membangun dan data yang kuat. Ini yang dinamakan aktivis sejati.' -> Prediksi: Positif
Komentar: 'Tidak bermanfaat sama sekali.' -> Prediksi: Negatif


4

In [11]:
# ==============================================================================
# 6. CONTOH PENGUJIAN KOMENTAR BARU
# ==============================================================================

def predict_sentiment(text_input, model, vectorizer):
    # 1. Preprocessing Teks Input
    clean_input = clean_text(text_input)
    # 2. Ekstraksi Fitur (menggunakan vectorizer yang sudah di-fit)
    vectorized_input = vectorizer.transform([clean_input])
    # 3. Prediksi
    prediction = model.predict(vectorized_input)[0]
    return prediction

print("\n--- Contoh Prediksi Komentar Baru ---")

# Komentar Positif
new_comment_positive = "Setiap kata-katanya penuh makna dan *insight* yang dalam."
predicted_label_pos = predict_sentiment(new_comment_positive, nb_model, tfidf_vectorizer)
print(f"Komentar: '{new_comment_positive}' -> Prediksi: {predicted_label_pos}")

# Komentar Negatif
new_comment_negative = "Tidak perlu ditonton. Buang-buang waktu."
predicted_label_neg = predict_sentiment(new_comment_negative, nb_model, tfidf_vectorizer)
print(f"Komentar: '{new_comment_negative}' -> Prediksi: {predicted_label_neg}")


--- Contoh Prediksi Komentar Baru ---
Komentar: 'Setiap kata-katanya penuh makna dan *insight* yang dalam.' -> Prediksi: Positif
Komentar: 'Tidak perlu ditonton. Buang-buang waktu.' -> Prediksi: Negatif
